# Wan 2.2 TI2V-5B Image-to-Video — free Colab T4 (16 GB)

Turns one product photo into a 9:16 MP4 with **Wan 2.2 TI2V-5B**, built for the **free** Colab T4 (about 15 GB VRAM, about 12.7 GB RAM). No High-RAM, L4, A100 or paid tier is needed.

**To run it:** `Runtime -> Run all`. The single code cell checks the GPU, asks for **one** product photo (the only click), installs, downloads the model (~34 GB, first run only), generates, plays the clip and downloads `output.mp4`. Nothing needs editing.

## Why this is not `generate.py`

The official `generate.py` builds the 10.6 GB text encoder and the 5B transformer in *system RAM* before anything reaches the GPU, so it needs roughly 20+ GiB of RAM. That cannot run on free Colab. This notebook uses the **official Diffusers integration of the same weights** (`Wan-AI/Wan2.2-TI2V-5B-Diffusers`, `WanImageToVideoPipeline`) and loads it in stages instead.

## Memory plan

| Technique | How it is used |
|---|---|
| Staged loading | The text encoder runs **alone** on the GPU (bf16), its embeddings are cached to disk, then it is released before the video model loads. Weights stream from disk to the GPU, so RAM stays low. |
| T5 CPU offload | If the encoder cannot fit the GPU alone, it falls back to accelerate CPU/disk offload. |
| fp16 transformer | About 9.3 GiB resident instead of 18.6 GiB in fp32 (a T4 has no fast bf16). Its norm/rope/time layers stay fp32 as the model requires. |
| Model / sequential CPU offload | Fallback settings: block-level and leaf-level group offloading of the transformer, used automatically when it cannot stay resident. |
| VAE slicing + tiling | Always on. The VAE runs in fp32, parked on the CPU during denoising, and decodes only after the transformer is freed (smaller tiles, then CPU, if it still runs out). |
| Attention | PyTorch SDPA memory-efficient kernel (probed at start); xFormers only if SDPA cannot use it. **Attention slicing** does not apply: the Wan transformer has no sliceable attention layers, and the log says so. |
| Automatic size/frame reduction | Each attempt runs in a fresh process. On out-of-memory it steps down: 576x1024 x 81 frames, 480x832 x 81, 480x832 x 49, then offload settings down to 288x512 x 33. |
| NaN guard | Latents are checked every step. If fp16 overflows, it retries in bf16 at a smaller size. |

Frame counts are always 4n+1 and sizes multiples of 32, as the model requires. Output is 24 fps.

## No runtime restart

Every heavy step (pip, the import check, the download, generation) runs in its own child process; the notebook kernel never imports `torch` or `diffusers`.

## Options

The prompt, negative prompt, steps, guidance and seed are form fields at the top of the cell. The defaults work as they are.

In [ ]:
#@title Wan 2.2 TI2V-5B image-to-video on a free T4  (Runtime > Run all)
# One cell does everything: checks the GPU, asks for ONE product photo, installs,
# downloads the model, generates the MP4 and downloads it. Nothing needs editing.
import json
import os
import shutil
import subprocess
import sys
import threading
import time

PROMPT = "Cinematic product advertisement: the product stays sharp and centered, slow smooth camera push-in, soft studio lighting, subtle natural motion, high quality commercial video"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, distorted, deformed, warped product, watermark, text, subtitles, jpeg artifacts, static, overexposed"  #@param {type:"string"}
NUM_STEPS = 30  #@param {type:"integer"}
GUIDANCE_SCALE = 5.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}

MODEL_REPO = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
ALLOW_PATTERNS = ["model_index.json", "scheduler/*", "tokenizer/*", "text_encoder/*", "transformer/*", "vae/*"]
MODEL_MIN_BYTES = 33 * 10 ** 9      # the three weight folders add up to about 34.2 GB
FPS = 24
BF16_MAX_TOKENS = 4500              # bf16 has no memory-efficient SDPA kernel on a T4 (sm75)

WORK = "/content" if os.path.isdir("/content") else os.getcwd()
MODEL_DIR = os.path.join(WORK, "Wan2.2-TI2V-5B-Diffusers")
WORKER_PATH = os.path.join(WORK, "wan_t4_worker.py")
EMBEDS_PATH = os.path.join(WORK, "prompt_embeds.pt")
OUTPUT = os.path.join(WORK, "output.mp4")

EXIT_OK, EXIT_OOM, EXIT_NAN, EXIT_UNSUPPORTED, EXIT_LOAD_OOM = 0, 3, 4, 5, 6

WORKER_SOURCE = r'''
import argparse
import gc
import hashlib
import os
import sys
import time
import traceback

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch

EXIT_OOM, EXIT_NAN, EXIT_UNSUPPORTED, EXIT_LOAD_OOM = 3, 4, 5, 6


class NanLatents(RuntimeError):
    pass


class LoadOOM(RuntimeError):
    pass


class Unsupported(RuntimeError):
    pass


def log(msg):
    print(time.strftime("[%H:%M:%S] ") + str(msg), flush=True)


def gib(nbytes):
    return nbytes / 2 ** 30


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def vram_report(tag):
    free, total = torch.cuda.mem_get_info()
    log("%s: VRAM in use %.1f / %.1f GiB" % (tag, gib(total - free), gib(total)))


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()


def transformers_dtype_kw(dtype):
    import transformers
    from packaging.version import Version
    key = "dtype" if Version(transformers.__version__) >= Version("4.56.0") else "torch_dtype"
    return {key: dtype}


# ---------------------------------------------------------------- stage 1: text
def encode_prompts(a):
    key = hashlib.sha1((a.prompt + "\n--\n" + a.negative).encode("utf-8")).hexdigest()
    if os.path.exists(a.embeds):
        try:
            cached = torch.load(a.embeds, map_location="cpu")
            if cached.get("key") == key:
                log("text embeddings: reusing the cached result (text encoder not loaded again)")
                return cached["pos"], cached["neg"]
        except Exception as exc:
            log("text embedding cache unreadable (%s); recomputing" % exc)

    from diffusers import UniPCMultistepScheduler, WanImageToVideoPipeline
    from transformers import AutoTokenizer, UMT5EncoderModel

    log("text encoder: loading UMT5-XXL (bf16) alone on the GPU; it is freed before the video model loads")
    tokenizer = AutoTokenizer.from_pretrained(os.path.join(a.model, "tokenizer"))
    kw = transformers_dtype_kw(torch.bfloat16)
    enc_dir = os.path.join(a.model, "text_encoder")
    try:
        encoder = UMT5EncoderModel.from_pretrained(enc_dir, device_map="cuda", **kw)
    except Exception as exc:
        log("text encoder did not fit on the GPU alone (%s); using accelerate CPU/disk offload" % type(exc).__name__)
        free_memory()
        encoder = UMT5EncoderModel.from_pretrained(
            enc_dir,
            device_map="auto",
            max_memory={0: "8GiB", "cpu": "5GiB"},
            offload_folder=os.path.join(a.work, "t5_offload"),
            **kw
        )
    encoder.eval()
    scheduler = UniPCMultistepScheduler.from_pretrained(a.model, subfolder="scheduler")
    pipe = WanImageToVideoPipeline(
        tokenizer=tokenizer,
        text_encoder=encoder,
        vae=None,
        scheduler=scheduler,
        transformer=None,
        expand_timesteps=True,
    )
    with torch.no_grad():
        pos, neg = pipe.encode_prompt(
            prompt=a.prompt,
            negative_prompt=a.negative,
            do_classifier_free_guidance=True,
            num_videos_per_prompt=1,
            max_sequence_length=512,
            device=torch.device("cuda"),
            dtype=torch.bfloat16,
        )
    pos, neg = pos.detach().cpu(), neg.detach().cpu()
    if not (torch.isfinite(pos.float()).all() and torch.isfinite(neg.float()).all()):
        raise RuntimeError("text encoder produced non-finite embeddings")
    torch.save({"key": key, "pos": pos, "neg": neg}, a.embeds)
    del pipe, encoder, tokenizer, scheduler
    free_memory()
    log("text encoder: done and released")
    return pos, neg


# ---------------------------------------------------------------- stage 2: video
def load_vae(model_dir):
    from diffusers import AutoencoderKLWan
    vae = AutoencoderKLWan.from_pretrained(model_dir, subfolder="vae", torch_dtype=torch.float32)
    for name in ("enable_slicing", "enable_tiling"):
        fn = getattr(vae, name, None)
        if fn is None:
            log("VAE: %s not available in this diffusers version" % name)
            continue
        fn()
        log("VAE: %s enabled" % name)
    return vae


def load_transformer(a, dtype):
    from diffusers import WanTransformer3DModel
    kw = dict(subfolder="transformer", torch_dtype=dtype)
    if a.offload == "none":
        try:
            model = WanTransformer3DModel.from_pretrained(a.model, device_map="cuda", **kw)
        except Exception as exc:
            if is_oom(exc):
                raise
            log("direct-to-GPU load unavailable (%s); loading on the CPU first" % type(exc).__name__)
            model = WanTransformer3DModel.from_pretrained(a.model, low_cpu_mem_usage=True, **kw).to("cuda")
    else:
        model = WanTransformer3DModel.from_pretrained(a.model, low_cpu_mem_usage=True, **kw)
        try:
            from diffusers.hooks import apply_group_offloading
        except Exception as exc:
            raise Unsupported("group offloading needs a newer diffusers: %s" % exc)
        onload, offload = torch.device("cuda"), torch.device("cpu")
        if a.offload == "group":
            apply_group_offloading(model, onload_device=onload, offload_device=offload,
                                   offload_type="block_level", num_blocks_per_group=2)
            log("transformer: block-level CPU offload (2 blocks resident at a time)")
        else:
            apply_group_offloading(model, onload_device=onload, offload_device=offload, offload_type="leaf_level")
            log("transformer: sequential (leaf-level) CPU offload")
    size = sum(p.numel() * p.element_size() for p in model.parameters())
    log("transformer: %.1f GiB of weights, mode=%s, dtype=%s" % (gib(size), a.offload, a.dtype))
    return model


def configure_attention(transformer, dtype):
    import torch.nn.functional as F
    from torch.nn.attention import SDPBackend, sdpa_kernel
    q = torch.randn(1, 4, 512, 64, device="cuda", dtype=dtype)
    try:
        with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            F.scaled_dot_product_attention(q, q, q)
        log("attention: PyTorch SDPA memory-efficient kernel is available and is used automatically")
        return
    except Exception as exc:
        log("attention: memory-efficient SDPA unavailable for %s (%s); trying xFormers" % (dtype, type(exc).__name__))
    try:
        import xformers.ops
        transformer.set_attention_backend("xformers")
        log("attention: xFormers %s backend enabled" % xformers.__version__)
    except Exception as exc:
        log("attention: xFormers not usable (%s); using math SDPA, keep the frame count small" % type(exc).__name__)


def prepare_image(path, width, height):
    from PIL import Image, ImageOps
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    return ImageOps.fit(img, (width, height), method=Image.LANCZOS, centering=(0.5, 0.5))


def generate_latents(a, pos, neg, dtype):
    from diffusers import UniPCMultistepScheduler, WanImageToVideoPipeline

    vae = load_vae(a.model).to("cuda")
    try:
        transformer = load_transformer(a, dtype)
    except Exception as exc:
        if is_oom(exc):
            raise LoadOOM(str(exc)) from exc
        raise
    configure_attention(transformer, dtype)
    scheduler = UniPCMultistepScheduler.from_pretrained(a.model, subfolder="scheduler")
    pipe = WanImageToVideoPipeline(
        tokenizer=None,
        text_encoder=None,
        vae=vae,
        scheduler=scheduler,
        transformer=transformer,
        expand_timesteps=True,
    )
    pipe.set_progress_bar_config(disable=True)
    sliceable = [n for n, m in pipe.components.items() if hasattr(m, "set_attention_slice")]
    if sliceable:
        pipe.enable_attention_slicing()
        log("attention slicing enabled on %s" % sliceable)
    else:
        log("attention slicing: not applicable (the Wan DiT has no sliceable attention); memory-efficient SDPA does that job")

    # The VAE is only needed for the first-frame encode inside the pipeline. Park it on the
    # CPU right afterwards so the GPU holds nothing but the transformer while denoising.
    original_prepare = pipe.prepare_latents

    def prepare_then_park_vae(*args, **kwargs):
        out = original_prepare(*args, **kwargs)
        vae.to("cpu")
        free_memory()
        return out

    pipe.prepare_latents = prepare_then_park_vae

    pos_d = pos.to("cuda", dtype)
    neg_d = neg.to("cuda", dtype)
    if not (torch.isfinite(pos_d).all() and torch.isfinite(neg_d).all()):
        raise NanLatents("prompt embeddings overflow in %s" % a.dtype)
    image = prepare_image(a.image, a.width, a.height)
    generator = torch.Generator(device="cuda").manual_seed(a.seed)
    started = time.time()

    def on_step(_pipe, index, _timestep, kwargs):
        lat = kwargs["latents"]
        if not bool(torch.isfinite(lat).all()):
            raise NanLatents("latents became NaN/inf at step %d" % (index + 1))
        done = index + 1
        elapsed = time.time() - started
        eta = elapsed / done * (a.steps - done)
        extra = ""
        if done == 1:
            extra = "  peak VRAM %.1f GiB" % gib(torch.cuda.max_memory_allocated())
        log("denoise step %d/%d  elapsed %ds  eta %ds%s" % (done, a.steps, elapsed, eta, extra))
        return kwargs

    log("denoising %dx%d, %d frames, %d steps, guidance %.1f" % (a.width, a.height, a.frames, a.steps, a.guidance))
    result = pipe(
        image=image,
        prompt=None,
        negative_prompt=None,
        prompt_embeds=pos_d,
        negative_prompt_embeds=neg_d,
        height=a.height,
        width=a.width,
        num_frames=a.frames,
        num_inference_steps=a.steps,
        guidance_scale=a.guidance,
        generator=generator,
        output_type="latent",
        callback_on_step_end=on_step,
        callback_on_step_end_tensor_inputs=["latents"],
    )
    latents = result.frames.detach()
    del pipe, transformer, result, pos_d, neg_d
    free_memory()
    vram_report("after denoising (transformer released)")
    return vae, latents


# ---------------------------------------------------------------- stage 3: decode
def decode_video(vae, latents):
    z_dim = vae.config.z_dim
    lat = latents.to(torch.float32)
    mean = torch.tensor(vae.config.latents_mean).view(1, z_dim, 1, 1, 1).to(lat.device, lat.dtype)
    inv_std = 1.0 / torch.tensor(vae.config.latents_std).view(1, z_dim, 1, 1, 1).to(lat.device, lat.dtype)
    lat = lat / inv_std + mean
    attempts = [("cuda", None), ("cuda", (128, 96)), ("cpu", (128, 96))]
    for where, tile in attempts:
        try:
            if tile is not None:
                vae.enable_tiling(
                    tile_sample_min_height=tile[0], tile_sample_min_width=tile[0],
                    tile_sample_stride_height=tile[1], tile_sample_stride_width=tile[1],
                )
            vae.to(where)
            log("VAE decode on %s%s" % (where, "" if tile is None else " with %dpx tiles" % tile[0]))
            with torch.no_grad():
                video = vae.decode(lat.to(where), return_dict=False)[0]
            return video.float().cpu()
        except Exception as exc:
            if not is_oom(exc):
                raise
            log("VAE decode ran out of memory on %s; retrying with a lighter setting" % where)
            free_memory()
    raise torch.cuda.OutOfMemoryError("VAE decode failed on every setting")


def save_video(vae, video, path, fps):
    from diffusers.video_processor import VideoProcessor
    if not bool(torch.isfinite(video).all()):
        raise NanLatents("decoded video contains NaN/inf")
    processor = VideoProcessor(vae_scale_factor=vae.config.scale_factor_spatial)
    frames = processor.postprocess_video(video, output_type="np")[0]
    if float(np.std(frames)) < 1e-3:
        raise NanLatents("decoded video is flat/black")
    log("encoding %d frames to MP4" % len(frames))
    try:
        from diffusers.utils import export_to_video
        export_to_video(list(frames), path, fps=fps)
    except Exception as exc:
        log("export_to_video failed (%s); writing with imageio" % exc)
        import imageio
        imageio.mimsave(path, [(f * 255).astype(np.uint8) for f in frames], fps=fps, macro_block_size=1)
    if not os.path.exists(path) or os.path.getsize(path) < 10000:
        raise RuntimeError("MP4 was not written")
    log("saved %s (%.1f MB)" % (path, os.path.getsize(path) / 1e6))


def run(a):
    if not torch.cuda.is_available():
        raise RuntimeError("no CUDA GPU is available")
    props = torch.cuda.get_device_properties(0)
    dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16}[a.dtype]
    log("GPU %s, %.1f GiB, torch %s, offload=%s, dtype=%s" % (props.name, gib(props.total_memory), torch.__version__, a.offload, a.dtype))
    pos, neg = encode_prompts(a)
    vae, latents = generate_latents(a, pos, neg, dtype)
    video = decode_video(vae, latents)
    del latents
    free_memory()
    save_video(vae, video, a.out, a.fps)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True)
    p.add_argument("--image", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--embeds", required=True)
    p.add_argument("--work", required=True)
    p.add_argument("--prompt", required=True)
    p.add_argument("--negative", required=True)
    p.add_argument("--width", type=int, required=True)
    p.add_argument("--height", type=int, required=True)
    p.add_argument("--frames", type=int, required=True)
    p.add_argument("--steps", type=int, required=True)
    p.add_argument("--guidance", type=float, required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--fps", type=int, default=24)
    p.add_argument("--offload", choices=["none", "group", "sequential"], default="none")
    p.add_argument("--dtype", choices=["float16", "bfloat16"], default="float16")
    a = p.parse_args()
    try:
        run(a)
    except NanLatents as exc:
        log("NAN: %s" % exc)
        return EXIT_NAN
    except LoadOOM:
        log("LOAD_OOM: the transformer does not fit in VRAM in this mode")
        return EXIT_LOAD_OOM
    except Unsupported as exc:
        log("UNSUPPORTED: %s" % exc)
        return EXIT_UNSUPPORTED
    except Exception as exc:
        if is_oom(exc):
            log("OOM: %s" % str(exc).splitlines()[0])
            return EXIT_OOM
        traceback.print_exc()
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

CHECK_CODE = r'''
import torch, transformers, diffusers, accelerate, imageio, ftfy, sentencepiece
from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanImageToVideoPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from diffusers.video_processor import VideoProcessor
from transformers import AutoTokenizer, UMT5EncoderModel
try:
    from diffusers.hooks import apply_group_offloading
    offload = "yes"
except Exception:
    offload = "NO"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| diffusers", diffusers.__version__,
      "| transformers", transformers.__version__, "| accelerate", accelerate.__version__, "| group offload", offload)
assert torch.cuda.is_available(), "torch cannot see the GPU"
'''


def stamp(msg):
    print(time.strftime("[%H:%M:%S] ") + msg, flush=True)


def run_streamed(cmd, env=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def run_captured(cmd):
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return proc.returncode, proc.stdout.strip()


def token_count(width, height, frames):
    return (width // 32) * (height // 32) * ((frames - 1) // 4 + 1)


def build_ladder(total_gib):
    """(width, height, frames, offload) from best to leanest. Sizes are multiples of 32, frames are 4n+1."""
    ladder = []
    if total_gib >= 22:
        ladder.append((704, 1280, 121, "none"))
    ladder += [
        (576, 1024, 81, "none"),
        (480, 832, 81, "none"),
        (480, 832, 49, "none"),
        (480, 832, 49, "group"),
        (352, 640, 49, "group"),
        (288, 512, 33, "sequential"),
    ]
    return ladder


def run_ladder(ladder, run_rung):
    """Walk the ladder. run_rung(width, height, frames, offload, dtype) -> exit code. Returns (rung, dtype) or None."""
    dtype = "float16"
    skip = set()
    i = 0
    while i < len(ladder):
        width, height, frames, offload = ladder[i]
        if offload in skip or (dtype == "bfloat16" and token_count(width, height, frames) > BF16_MAX_TOKENS):
            i += 1
            continue
        code = run_rung(width, height, frames, offload, dtype)
        if code == EXIT_OK:
            return ladder[i], dtype
        if code == EXIT_OOM:
            stamp("out of GPU memory at %dx%d x %d frames (%s) -> trying a lighter setting" % (width, height, frames, offload))
            i += 1
        elif code == EXIT_LOAD_OOM:
            stamp("the transformer does not fit resident on this GPU -> switching to CPU-offload settings")
            skip.add("none")
            i += 1
        elif code == EXIT_UNSUPPORTED:
            stamp("CPU-offload is not supported by the installed diffusers -> skipping offload settings")
            skip.update(("group", "sequential"))
            i += 1
        elif code == EXIT_NAN and dtype == "float16":
            stamp("fp16 overflowed (NaN) -> retrying in bf16 at a size its attention path can hold")
            dtype = "bfloat16"
        else:
            stamp("generation failed with exit code %s (details above)" % code)
            return None
    return None


def verify_checkpoint(model_dir):
    """Return a list of problems (empty when every file the model needs is present)."""
    problems = []
    total = 0
    for sub, index_name, single in (
        ("transformer", "diffusion_pytorch_model.safetensors.index.json", None),
        ("text_encoder", "model.safetensors.index.json", None),
        ("vae", None, "diffusion_pytorch_model.safetensors"),
    ):
        folder = os.path.join(model_dir, sub)
        files = [single] if single else []
        if index_name:
            index_path = os.path.join(folder, index_name)
            if not os.path.exists(index_path):
                problems.append("missing " + sub + "/" + index_name)
            else:
                with open(index_path, "r", encoding="utf-8") as fh:
                    files = sorted(set(json.load(fh)["weight_map"].values()))
        for name in files:
            path = os.path.join(folder, name)
            if not os.path.exists(path) or os.path.getsize(path) == 0:
                problems.append("missing " + sub + "/" + name)
            else:
                total += os.path.getsize(path)
    for name in ("model_index.json", "scheduler/scheduler_config.json", "tokenizer/spiece.model", "tokenizer/tokenizer.json"):
        if not os.path.exists(os.path.join(model_dir, name)):
            problems.append("missing " + name)
    if not problems and total < MODEL_MIN_BYTES:
        problems.append("weights total only %.1f GB" % (total / 1e9))
    return problems


def dir_size(path):
    total = 0
    for root, _dirs, names in os.walk(path):
        for name in names:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass
    return total


def main():
    started = time.time()
    stamp("Step 1/6  checking the runtime")
    code, smi = run_captured(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"])
    if code != 0 or not smi:
        raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
    name, total_mib, used_mib = [part.strip() for part in smi.splitlines()[0].split(",")]
    total_gib = float(total_mib) / 1024
    print("GPU: %s | %.1f GiB VRAM (%s MiB already in use)" % (name, total_gib, used_mib))
    ram_gib = None
    try:
        with open("/proc/meminfo") as fh:
            ram_gib = int(fh.readline().split()[1]) / 1024 / 1024
    except (OSError, ValueError, IndexError):
        pass
    print("System RAM: %s | pipeline is staged to fit in it" % ("unknown" if ram_gib is None else "%.1f GiB" % ram_gib))
    have_model = os.path.isdir(MODEL_DIR) and not verify_checkpoint(MODEL_DIR)
    if not have_model and shutil.disk_usage(WORK).free < 40 * 10 ** 9:
        raise SystemExit("Not enough free disk (need ~40 GB for the model download). Runtime > Disconnect and delete runtime, then Run all.")

    stamp("Step 2/6  upload ONE product photo (the only thing you need to click)")
    image_path = None
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            original = sorted(uploaded)[0]
            ext = os.path.splitext(original)[1].lower() or ".jpg"
            image_path = os.path.join(WORK, "input_image" + ext)
            with open(image_path, "wb") as fh:
                fh.write(uploaded[original])
    except ImportError:
        pass
    if image_path is None:
        older = [os.path.join(WORK, f) for f in os.listdir(WORK) if f.startswith("input_image.")]
        if not older:
            raise SystemExit("No image was uploaded. Run the cell again and choose a product photo.")
        image_path = older[0]
        print("no new upload - reusing", image_path)
    from PIL import Image
    with Image.open(image_path) as probe:
        probe.verify()
    print("image ok:", image_path)

    stamp("Step 3/6  installing dependencies (runs in child processes - no restart needed)")
    pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    steps = [
        ["-U", "diffusers", "accelerate", "ftfy", "sentencepiece", "protobuf", "imageio", "imageio-ffmpeg"],
        ["-U", "transformers"],
        ["-U", "git+https://github.com/huggingface/diffusers"],
    ]
    ready = False
    for extra in steps:
        if run_streamed(pip + extra) != 0:
            print("pip step failed; trying the next option")
            continue
        code, out = run_captured([sys.executable, "-c", CHECK_CODE])
        print(out.splitlines()[-1] if out else "")
        if code == 0:
            ready = True
            break
    if not ready:
        raise SystemExit("Dependencies could not be made importable. The last message is printed above.")
    if "group offload NO" in out:
        print("note: this diffusers has no group offloading; only in-VRAM settings will be attempted")

    stamp("Step 4/6  downloading the model (~34 GB, resumable)")
    for attempt in (1, 2):
        problems = verify_checkpoint(MODEL_DIR) if os.path.isdir(MODEL_DIR) else ["not downloaded"]
        if not problems:
            print("model files are all present")
            break
        if attempt == 2:
            raise SystemExit("Model download is incomplete: " + "; ".join(problems))
        stop = threading.Event()

        def monitor():
            while not stop.wait(20):
                print("   downloaded %.1f GB of ~34 GB" % (dir_size(MODEL_DIR) / 1e9), flush=True)

        threading.Thread(target=monitor, daemon=True).start()
        download_code = (
            "import os, sys, time\n"
            "os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'\n"
            "from huggingface_hub import snapshot_download\n"
            "for n in (1, 2, 3):\n"
            "    try:\n"
            "        snapshot_download(repo_id=%r, local_dir=%r, allow_patterns=%r, max_workers=8)\n"
            "        sys.exit(0)\n"
            "    except Exception as exc:\n"
            "        print('download attempt', n, 'failed:', type(exc).__name__, exc, flush=True)\n"
            "        time.sleep(5)\n"
            "sys.exit(1)\n"
        ) % (MODEL_REPO, MODEL_DIR, ALLOW_PATTERNS)
        run_streamed([sys.executable, "-c", download_code])
        stop.set()

    stamp("Step 5/6  generating (T4 memory plan: staged loading, fp16, CPU offload, VAE slicing+tiling, SDPA)")
    with open(WORKER_PATH, "w", encoding="utf-8") as fh:
        fh.write(WORKER_SOURCE)
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", PYTHONUNBUFFERED="1")

    def run_rung(width, height, frames, offload, dtype):
        stamp("attempt: %dx%d, %d frames (%.1fs), offload=%s, %s" % (width, height, frames, frames / FPS, offload, dtype))
        cmd = [
            sys.executable, WORKER_PATH,
            "--model", MODEL_DIR, "--image", image_path, "--out", OUTPUT, "--embeds", EMBEDS_PATH, "--work", WORK,
            "--prompt=" + PROMPT, "--negative=" + NEGATIVE_PROMPT,
            "--width", str(width), "--height", str(height), "--frames", str(frames),
            "--steps", str(NUM_STEPS), "--guidance", str(GUIDANCE_SCALE), "--seed", str(SEED), "--fps", str(FPS),
            "--offload", offload, "--dtype", dtype,
        ]
        return run_streamed(cmd, env=env)

    outcome = run_ladder(build_ladder(total_gib), run_rung)
    if outcome is None or not os.path.exists(OUTPUT):
        raise SystemExit("No setting produced a video. Read the messages above the last attempt; "
                         "if the GPU shows memory in use, Runtime > Disconnect and delete runtime, then Run all.")
    (width, height, frames, offload), dtype = outcome
    stamp("Step 6/6  done: %dx%d, %d frames, offload=%s, %s, total %.1f min" % (width, height, frames, offload, dtype, (time.time() - started) / 60))
    try:
        from IPython.display import Video, display
        display(Video(OUTPUT, embed=True, width=320))
    except Exception:
        pass
    try:
        from google.colab import files
        files.download(OUTPUT)
    except ImportError:
        print("saved to", OUTPUT)


main()